# Data Preparation

This notebook documents our full data preparation process for the **Chicago Taxi Trips 2024** dataset. Rather than presenting a clean pipeline upfront, we work through the data step by step: examining the raw data first, surfacing issues, and justifying each cleaning decision as we find it.

All cleaning steps documented here are encapsulated in `scripts/helpers/preprocessing.py` and are applied automatically when subsequent notebooks call `load_taxi_data(preprocessed=True)`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from pathlib import Path
import sys
import os

ROOT_DIR = Path().resolve().parent
sys.path.insert(0, str(ROOT_DIR))

from scripts.helpers.datasets import load_taxi_data

## First Look at the Raw Data

We load the dataset with `preprocessed=False` to see exactly what the Chicago Data Portal delivered — before any cleaning is applied. This gives us a honest baseline to reason from.

In [2]:
df = load_taxi_data(preprocessed=False)

In [3]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 6593828 entries, 0 to 6593827
Data columns (total 21 columns):
 #   Column                      Non-Null Count    Dtype                          
---  ------                      --------------    -----                          
 0   trip_id                     6593828 non-null  str                            
 1   taxi_id                     6593824 non-null  str                            
 2   trip_start_timestamp        6593724 non-null  datetime64[us, America/Chicago]
 3   trip_end_timestamp          6593717 non-null  datetime64[us, America/Chicago]
 4   trip_seconds                6593828 non-null  int64                          
 5   trip_miles                  6593828 non-null  float64                        
 6   pickup_census_tract         6593828 non-null  int64                          
 7   dropoff_census_tract        6593828 non-null  int64                          
 8   pickup_community_area       6586290 non-null  float64          

We're working with **6.6 million trips** across **21 columns**.

Let's peek at the actual records before diving into cleaning.

In [5]:
df.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude
0,22208de57c456e7e6ea5f60bdc1746ad300535a9,04b96cbbdcfe5b7cbb6884bc1b922819466f652662ead8...,2024-01-01 00:00:00-06:00,2024-01-01 00:30:00-06:00,2214,18.23,17031980000,17031320100,76.0,32.0,...,7.91,0.0,4.0,60.66,Credit Card,5 Star Taxi,41.979071,-87.903040,41.884987,-87.620993
1,35968e44a8ea32a0849720b91c35a4d5a8ff6484,4a991432c3e0600b9c919a01148b17b866d29a41751b95...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,120,0.00,17031081600,17031081201,8.0,8.0,...,0.00,0.0,0.0,3.75,Cash,Taxi Affiliation Services,41.892073,-87.628874,41.899156,-87.626211
2,3c05ccf0732fc338b7c875f9a9779039eaada274,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,2024-01-01 00:00:00-06:00,2024-01-01 00:30:00-06:00,1681,15.34,17031980000,17031071400,76.0,7.0,...,8.85,0.0,4.0,53.10,Credit Card,Globe Taxi,41.979071,-87.903040,41.922083,-87.634156
3,541cf2b862280d13b36e466ad90d9485e1ae1600,13c8f729e7e5a9f850e406e3b31524c6881649044dab76...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,59,0.00,17031980000,17031980000,76.0,76.0,...,0.00,0.0,0.0,3.50,Cash,5 Star Taxi,41.979071,-87.903040,41.979071,-87.903040
4,63d8c865c01bde9e17e469db6a30e33c8cfe5314,259d38cfdbc9ac6f9bb01f0df740e0ddf4a631a70bbdd6...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,180,0.30,17031081500,17031081201,8.0,8.0,...,0.00,0.0,1.0,5.25,Cash,"Taxicab Insurance Agency, LLC",41.892508,-87.626215,41.899156,-87.626211


A few observations from the first rows:

- `trip_id` and `taxi_id` are SHA-1 hashes — no personally identifiable information
- Timestamps are **rounded to 15-minute intervals** (a privacy feature of the Chicago Data Portal)
- Several trips in the first 5 rows have `trip_seconds > 0` but `trip_miles == 0` — worth investigating

Let's confirm the full column list, then start tackling the missing values.

In [6]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'dropoff_centroid_latitude',
       'dropoff_centroid_longitude'],
      dtype='str')

## Handling Missing Values

With ~6.6 M rows, even small missingness rates can represent tens of thousands of trips. We need to understand the extent and cause of each gap before deciding how to handle it.

In [7]:
print(df.isnull().sum())

trip_id                           0
taxi_id                           4
trip_start_timestamp            104
trip_end_timestamp              111
trip_seconds                      0
trip_miles                        0
pickup_census_tract               0
dropoff_census_tract              0
pickup_community_area          7538
dropoff_community_area        80242
fare                              0
tips                              0
tolls                             0
extras                            0
trip_total                        0
payment_type                      0
company                           0
pickup_centroid_latitude          0
pickup_centroid_longitude         0
dropoff_centroid_latitude         0
dropoff_centroid_longitude        0
dtype: int64


Three types of missingness stand out:

1. **Community areas**: 7,538 missing pickups and 80,242 missing dropoffs. These are the most important to recover since community area is central to our spatial analysis. Crucially, the centroid coordinates (`pickup_centroid_latitude/longitude`) are always populated, so we can recover the community area via a spatial join against Chicago's official boundary polygons.

2. **Timestamps**: 104–111 missing entries (<0.002% of trips). Tiny in volume, but we want to understand *why* before deciding to drop them.

3. **Taxi ID**: only 4 missing entries. We will drop these since no vehicle can be linked to the trip.

### Missing Timestamps

Let's inspect the raw timestamp strings for the rows that became `NaT` after parsing; if they share a date, that's a strong clue.

In [8]:
RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
raw_ts = pd.read_csv(RAW_DATA_DIR / "chicago_taxi_trips_2024.csv", 
                      usecols=["trip_start_timestamp"])
nat_rows = df[df["trip_start_timestamp"].isna()].index
print(raw_ts.loc[nat_rows, "trip_start_timestamp"].unique())

<StringArray>
['2024-11-03T01:00:00.000', '2024-11-03T01:15:00.000',
 '2024-11-03T01:30:00.000', '2024-11-03T01:45:00.000',
 '2025-11-02T01:00:00.000', '2025-11-02T01:15:00.000',
 '2025-11-02T01:30:00.000', '2025-11-02T01:45:00.000']
Length: 8, dtype: str


The missing timestamps coincide with the daylight saving time transition from summer to winter time. Rather than being truly absent, these entries were simply dropped during parsing because they were ambiguous or invalid in that context. AS this is the case for a very small portion of trips, we can just drop these observations.

### Missing Community Areas
Chicago census tracts are fully nested within community areas, so every trip coordinate maps to exactly one community area. We use a spatial join against the official Chicago community area boundaries (downloaded via `scripts/download_community_areas.py`) to fill the missing values. Trips whose coordinates fall outside Chicago's 77 community areas (e.g. suburban drop-offs) will remain NaN and are kept as-is.

In [11]:
ca_gdf = gpd.read_file(ROOT_DIR / "data" / "raw" / "community_areas.geojson")[["area_numbe", "geometry"]]
ca_gdf.rename(columns={"area_numbe": "area_number"}, inplace=True)
ca_gdf["area_number"] = pd.to_numeric(ca_gdf["area_number"])
ca_gdf.head()

,area_number,geometry
0,1,"MULTIPOLYGON (((-87.65456 41.99817, -87.65574 ..."
1,2,"MULTIPOLYGON (((-87.68465 42.01948, -87.68464 ..."
2,3,"MULTIPOLYGON (((-87.64102 41.9548, -87.644 41...."
3,4,"MULTIPOLYGON (((-87.67441 41.9761, -87.6744 41..."
4,5,"MULTIPOLYGON (((-87.67336 41.93234, -87.67342 ..."


In [12]:
def fill_community_area(df, lat_col, lon_col, ca_col, ca_gdf):
    null_mask = df[ca_col].isna()
    if not null_mask.any():
        return df

    pts = gpd.GeoDataFrame(
        index=df.index[null_mask],
        geometry=gpd.points_from_xy(
            df.loc[null_mask, lon_col],
            df.loc[null_mask, lat_col],
        ),
        crs="EPSG:4326",
    )
    joined = pts.sjoin(ca_gdf, how="left", predicate="within")
    df.loc[null_mask, ca_col] = joined["area_number"].values
    return df

df = fill_community_area(df, "pickup_centroid_latitude",  "pickup_centroid_longitude",  "pickup_community_area",  ca_gdf)
df = fill_community_area(df, "dropoff_centroid_latitude", "dropoff_centroid_longitude", "dropoff_community_area", ca_gdf)

print(df[["pickup_community_area", "dropoff_community_area"]].isnull().sum())

pickup_community_area     0
dropoff_community_area    0
dtype: int64


### Missing Taxi ID

Only 4 trips have no `taxi_id`. Since we cannot associate these trips with any vehicle they are useless for fleet-level analysis we drop them in `preprocess_taxi_data`. The updated null check confirms the community area imputation succeeded and no other columns gained new gaps:

In [13]:
print(df.isnull().sum())

trip_id                         0
taxi_id                         4
trip_start_timestamp          104
trip_end_timestamp            111
trip_seconds                    0
trip_miles                      0
pickup_census_tract             0
dropoff_census_tract            0
pickup_community_area           0
dropoff_community_area          0
fare                            0
tips                            0
tolls                           0
extras                          0
trip_total                      0
payment_type                    0
company                         0
pickup_centroid_latitude        0
pickup_centroid_longitude       0
dropoff_centroid_latitude       0
dropoff_centroid_longitude      0
dtype: int64


The only remaining nulls are the ~215 DST-ambiguous timestamps and the 4 `taxi_id` rows — both handled by `preprocess_taxi_data`. Everything else is clean.

## Consistency Checks

Null counts only catch *absent* values — not *implausible* ones. We now look for trips that are technically present but don't represent real journeys: zero-duration ghost trips, zero-distance meter errors, and end-before-start timestamp inversions.

### Ghost Trips (Zero Duration, Same Location)

A trip where `trip_seconds == 0`, start equals end timestamp, *and* pickup equals dropoff coordinates is indistinguishable from a meter accidentally switched on and immediately off. These rows carry no mobility signal and would inflate trip counts in any temporal or spatial aggregation.

In [14]:
mask = (
    (df['trip_seconds'] == 0) &
    (df['trip_end_timestamp'] == df['trip_start_timestamp']) &
    (df['pickup_centroid_latitude'] == df['dropoff_centroid_latitude']) &
    (df['pickup_centroid_longitude'] == df['dropoff_centroid_longitude'])
)
mask.sum()

np.int64(83221)

**83,221 ghost trips** — roughly 1.3% of the dataset. These are dropped by `preprocess_taxi_data` in `scripts/helpers/preprocessing.py`. The criterion is strict: all four conditions must hold simultaneously, so genuine very-short trips that covered even a small distance are preserved.

### Trips with 0 Miles but Positive Duration

Ghost trips covered the most obvious no-movement cases. There is a subtler variant: trips where `trip_seconds > 0` (the meter ran) but `trip_miles == 0` (no distance recorded). This typically indicates a GPS failure — the cab moved, but coordinates never updated. We look at the duration distribution to get a sense of what these trips look like.

In [15]:
df_zero_miles = df[(df['trip_miles'] == 0) & (df['trip_seconds'] > 0)]
print(df_zero_miles['trip_seconds'].describe())

count    496082.000000
mean        478.124457
std        1399.234288
min           1.000000
25%           6.000000
50%          46.000000
75%         511.000000
max       86396.000000
Name: trip_seconds, dtype: float64


The 496,082 trips with `trip_miles == 0` and `trip_seconds > 0` are dropped. The duration distribution reveals that the vast majority are not real trips: the median duration is only 46 seconds and the 25tgh percentile is just 6 seconds, indicating these are overwhelmingly meter-on/meter-off cancellations or GPS failures where movement was never recorded. A real taxitrip must cover some distance; retaining these rows would distort trip distance and speed features and corrupt spatial demand analysis since pickup and dropoff centroids coincide for GPS failures.

In [54]:
df = df[df['trip_miles'] > 0]
print(df.shape)

(6012652, 21)


The filter removed ~580k rows ($\approx8.8\%$ of the original dataset), leaving **6,012,652 trips**. The cells below are verification checks from our exploration — they confirm the pre-filter counts and cross-check the mask logic.

In [11]:
mask_2 = (
    df['trip_miles'] == 0
)
mask_2.sum()

np.int64(575888)

In [12]:
mask_2 = (
    (df['trip_miles'] <= 0) &
    (df['trip_seconds'] > 0)
)
mask_2.sum()


np.int64(496082)

### End Timestamp Before Start Timestamp

A final sanity check: can a trip's recorded end time precede its start time? This would indicate a clock error or data entry mistake, and any duration or speed derived from such rows would be nonsensical.

In [13]:
mask_3 = (
    df['trip_end_timestamp'] < df['trip_start_timestamp']
)
mask_3.sum()

np.int64(0)

No inverted timestamps — the dataset is self-consistent throughout. No action needed here.

## Handling Duplicates

### Duplicate Trip IDs

Each row should represent a unique journey, identified by `trip_id`. Duplicate IDs would mean the same trip is counted multiple times — inflating demand figures and revenue totals. We check for any `trip_id` that appears more than once.

In [14]:
dup_df = df[df['trip_id'].duplicated(keep=False)]
print(f"Number of duplicate trip_id entries:", dup_df.shape[0])

Number of duplicate trip_id entries: 0


Zero duplicates — `trip_id` is a clean primary key across the full dataset. No deduplication needed.

## Loading & Preprocessing in Subsequent Notebooks

All steps documented above are encapsulated in two helper scripts so that later notebooks can load a clean dataset in a single line:

- **`scripts/helpers/datasets.py`** — entry point for loading any dataset. `load_taxi_data(preprocessed=True)` reads the raw CSV and optionally applies the full preprocessing pipeline.
- **`scripts/helpers/preprocessing.py`** — contains `preprocess_taxi_data()`, which imputes missing community areas, drops invalid rows, and returns a clean `DataFrame`.

Usage in any subsequent notebook:

```python
from scripts.helpers.datasets import load_taxi_data, load_merged_data

df = load_taxi_data(preprocessed=True)       # cleaned dataset
df_raw = load_taxi_data(preprocessed=False)  # raw data only
df_merged = load_merged_data() # preprocessed taxi + weather data
```